# Experiment 9: Chunking (Shallow Parsing)

**Mapped CO:** CO5  
**BT Level:** BT5

## Aim
To perform shallow parsing by identifying noun phrases and other non-overlapping chunks in POS-tagged text.

## Objectives
- Understand chunking as shallow syntactic analysis.
- Use POS tags as input features for a rule-based chunk grammar.
- Build noun-phrase chunk patterns with regular-expression grammar.
- Inspect IOB-style chunk structure and parse-tree output.

## Requirements
Python 3.10 or above, `nltk`, an NLTK tokenizer and POS-tagger resource, and a Jupyter notebook environment.

## Theory

Chunking, also called shallow parsing, groups sequences of tokens into flat phrase units without building a complete syntactic tree. Typical chunks include noun phrases (NP), verb phrases, and prepositional phrases. Chunking normally begins after tokenization and POS tagging.

An NLTK `RegexpParser` applies rules such as an optional determiner followed by adjectives and one or more nouns. In IOB notation, **B** marks the beginning of a chunk, **I** marks a continuation, and **O** marks a token outside a chunk. Standard shallow parsing produces non-overlapping, generally non-nested chunks.

## Methodology
1. Tokenize and POS-tag an input sentence.
2. Define a noun-phrase chunk grammar.
3. Create an NLTK `RegexpParser`.
4. Parse the tagged sentence.
5. Print the resulting chunk tree.
6. Convert the tree to IOB tags for inspection.

In [1]:
import nltk
from nltk import pos_tag, word_tokenize
from nltk.chunk import RegexpParser

REQUIRED_RESOURCES = {
    "punkt": "tokenizers/punkt",
    "punkt_tab": "tokenizers/punkt_tab",
    "averaged_perceptron_tagger": "taggers/averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng": "taggers/averaged_perceptron_tagger_eng",
}

def ensure_nltk_resources():
    """Download missing NLTK resources and report failures explicitly."""
    missing = []
    for package, resource_path in REQUIRED_RESOURCES.items():
        try:
            nltk.data.find(resource_path)
        except LookupError:
            missing.append(package)

    for package in missing:
        print(f"Downloading missing NLTK resource: {package}")
        if not nltk.download(package, quiet=False):
            raise RuntimeError(
                f"NLTK could not download {package}. Install it manually and rerun."
            )

ensure_nltk_resources()
print("NLTK resources are ready.")

NLTK resources are ready.


## Dataset and input sentence

The demonstration uses an English sentence containing determiners, adjectives, nouns, verbs, and a prepositional phrase. The optional evaluation cell uses the CoNLL-2000 chunking corpus.

In [2]:
sentence = (
    "The intelligent student built a powerful language model "
    "for natural language processing."
)
tokens = word_tokenize(sentence)
tagged_sentence = pos_tag(tokens)
print("Sentence:", sentence)
print("POS-tagged tokens:")
print(tagged_sentence)

Sentence: The intelligent student built a powerful language model for natural language processing.
POS-tagged tokens:
[('The', 'DT'), ('intelligent', 'JJ'), ('student', 'NN'), ('built', 'VBD'), ('a', 'DT'), ('powerful', 'JJ'), ('language', 'NN'), ('model', 'NN'), ('for', 'IN'), ('natural', 'JJ'), ('language', 'NN'), ('processing', 'NN'), ('.', '.')]


In [3]:
grammar = r"""
    NP: {<DT>?<JJ.*>*<NN.*>+}
"""
chunker = RegexpParser(grammar)
chunk_tree = chunker.parse(tagged_sentence)

print("Chunk grammar:")
print(grammar)
print("NP chunk tree:")
print(chunk_tree)
print("\nPretty tree representation:")
print(chunk_tree.pformat(margin=100))

Chunk grammar:

    NP: {<DT>?<JJ.*>*<NN.*>+}

NP chunk tree:
(S
  (NP The/DT intelligent/JJ student/NN)
  built/VBD
  (NP a/DT powerful/JJ language/NN model/NN)
  for/IN
  (NP natural/JJ language/NN processing/NN)
  ./.)

Pretty tree representation:
(S
  (NP The/DT intelligent/JJ student/NN)
  built/VBD
  (NP a/DT powerful/JJ language/NN model/NN)
  for/IN
  (NP natural/JJ language/NN processing/NN)
  ./.)


In [5]:
from nltk.chunk.util import tree2conlltags

iob_rows = tree2conlltags(chunk_tree)

print(f"{'Token':<18} {'POS':<8} {'IOB':<6}")
print("-" * 34)

for token, pos, iob in iob_rows:
    print(f"{token:<18} {pos:<8} {iob:<6}")

Token              POS      IOB   
----------------------------------
The                DT       B-NP  
intelligent        JJ       I-NP  
student            NN       I-NP  
built              VBD      O     
a                  DT       B-NP  
powerful           JJ       I-NP  
language           NN       I-NP  
model              NN       I-NP  
for                IN       O     
natural            JJ       B-NP  
language           NN       I-NP  
processing         NN       I-NP  
.                  .        O     


In [6]:
second_sentence = "Several experienced researchers designed an efficient neural network."
second_tagged = pos_tag(word_tokenize(second_sentence))
second_tree = chunker.parse(second_tagged)
print("POS-tagged tokens:", second_tagged)
print("Chunk tree:", second_tree.pformat(margin=100))
print("IOB labels:", tree2conlltags(second_tree))

POS-tagged tokens: [('Several', 'JJ'), ('experienced', 'VBD'), ('researchers', 'NNS'), ('designed', 'VBN'), ('an', 'DT'), ('efficient', 'JJ'), ('neural', 'JJ'), ('network', 'NN'), ('.', '.')]
Chunk tree: (S
  Several/JJ
  experienced/VBD
  (NP researchers/NNS)
  designed/VBN
  (NP an/DT efficient/JJ neural/JJ network/NN)
  ./.)
IOB labels: [('Several', 'JJ', 'O'), ('experienced', 'VBD', 'O'), ('researchers', 'NNS', 'B-NP'), ('designed', 'VBN', 'O'), ('an', 'DT', 'B-NP'), ('efficient', 'JJ', 'I-NP'), ('neural', 'JJ', 'I-NP'), ('network', 'NN', 'I-NP'), ('.', '.', 'O')]


## Optional evaluation with CoNLL-2000

The CoNLL-2000 corpus contains gold chunk labels. The following cell evaluates the same noun-phrase grammar when the corpus is installed. If it is unavailable, the cell reports the exact resource needed instead of failing silently.

In [7]:
try:
    from nltk.corpus import conll2000
    from nltk.chunk import ChunkScore

    test_sentences = conll2000.chunked_sents("test.txt", chunk_types=["NP"])
    score = ChunkScore()
    for gold_tree in test_sentences:
        predicted_tree = chunker.parse([(word, tag) for word, tag, _ in gold_tree.leaves()])
        score.score(gold_tree, predicted_tree)

    print(f"Evaluated sentences: {len(test_sentences)}")
    print(f"Precision: {score.precision():.4f}")
    print(f"Recall:    {score.recall():.4f}")
    print(f"F-measure: {score.f_measure():.4f}")
except LookupError:
    print("CoNLL-2000 is not installed. Run nltk.download('conll2000') and rerun this cell.")

CoNLL-2000 is not installed. Run nltk.download('conll2000') and rerun this cell.


## Results
The output contains POS-tagged tokens, an NP chunk tree, and IOB labels. Noun phrases such as **The intelligent student** and **a powerful language model** are grouped according to the grammar.

## Outcomes
- Apply shallow parsing to POS-tagged text.
- Write POS-pattern chunk grammars.
- Interpret chunk trees and IOB labels.
- Compare chunking with full syntactic parsing.

## Resources
- [NLTK Book: Chapter 7](https://www.nltk.org/book/ch07.html)
- [NLTK chunk API](https://www.nltk.org/api/nltk.chunk.html)
- [Speech and Language Processing](https://web.stanford.edu/~jurafsky/slp3/)
- [Demonstrative video search](https://www.youtube.com/results?search_query=NPTEL+chunking+shallow+parsing+IOB+NLP)

## Self-assessment questions and answers

1. **Define chunking or shallow parsing.** It groups adjacent POS-tagged tokens into flat, non-overlapping phrase units without constructing a complete parse tree.
2. **How does chunking differ from full parsing?** Chunking identifies local phrases, whereas full parsing represents hierarchical grammatical relationships for the whole sentence.
3. **What is a noun phrase chunk?** A sequence centered on a noun, optionally including determiners and modifiers such as adjectives.
4. **Why is POS tagging performed before chunking?** POS tags provide the grammatical features used by rule-based chunk patterns.
5. **Explain B, I, and O labels.** B begins a chunk, I continues the current chunk, and O marks a token outside any chunk.
6. **What does `<DT>?<JJ.*>*<NN.*>+` represent?** An optional determiner, followed by zero or more adjective tags, and one or more noun tags.
7. **What is `RegexpParser`?** An NLTK rule-based parser that applies regular-expression chunk grammar rules to tagged tokens.
8. **Can chunks be nested in standard shallow parsing output?** No. Standard chunk output is flat and non-overlapping; nested structure requires a full parser or another representation.
9. **Name two applications of chunking.** Information extraction and question answering; it is also useful in named-entity preprocessing and document indexing.
10. **How would you evaluate a chunker?** Compare predicted chunks with gold-standard chunks using precision, recall, and F-measure, as in the CoNLL-2000 evaluation.